In [ ]:
#####intall 一些package包

In [21]:
!pip install osmnx geopandas pandas

In [2]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

In [ ]:
#####现在jupyter的工作路径在哪

In [3]:
import os
print(os.getcwd())

/home/jovyan/work/DSSS_assessment


In [ ]:
#####cd 文件夹

In [18]:
%cd /home/jovyan/work/DSSS_assessment

/home/jovyan/work/DSSS_assessment


In [5]:
import os
print(os.getcwd())
print(os.listdir())

/home/jovyan/work/DSSS_assessment
['CASA0006_AssessmentGuidelines_2025.pdf', '.DS_Store', 'Template_submission_CASA0006.ipynb', 'cache', '助教.docx', 'dataset', 'CASA0006_CourseworkMark_Scheme_2025.pdf', '.ipynb_checkpoints', '.git', 'test.ipynb', 'reference']


In [6]:
print(os.listdir("dataset"))

['.DS_Store', 'MPS LSOA Level Crime (most recent 24 months).csv', 'population', 'workplacepopulation_population exposure2', 'oproad_essh_gb', 'London_boundaries', 'Lower_layer_Super_Output_Areas', 'File_1_IoD2025_Index_of_Multiple_Deprivation.xlsx', 'File_1_-_IMD2019_Index_of_Multiple_Deprivation.xlsx', 'MPS LSOA Level Crime (Historical).csv']


In [7]:
import os

print(os.listdir("dataset/London_boundaries"))

['LSOA_2011_London_gen_MHW.shx', 'LSOA_2011_London_gen_MHW.shp', 'LSOA_2011_London_gen_MHW.dbf', 'LSOA_2011_London_gen_MHW.sbn', 'LSOA_2011_London_gen_MHW.prj']


In [ ]:
#####第一个读取本地raw文件的地方，读取london的boundary

In [8]:
import geopandas as gpd

lsoa = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp")

print(lsoa.head())
print(len(lsoa))

    LSOA11CD                   LSOA11NM   MSOA11CD                  MSOA11NM  \
0  E01000001        City of London 001A  E02000001        City of London 001   
1  E01000002        City of London 001B  E02000001        City of London 001   
2  E01000003        City of London 001C  E02000001        City of London 001   
3  E01000005        City of London 001E  E02000001        City of London 001   
4  E01000006  Barking and Dagenham 016A  E02000017  Barking and Dagenham 016   

     LAD11CD               LAD11NM    RGN11CD RGN11NM  USUALRES  HHOLDRES  \
0  E09000001        City of London  E12000007  London      1465      1465   
1  E09000001        City of London  E12000007  London      1436      1436   
2  E09000001        City of London  E12000007  London      1346      1250   
3  E09000001        City of London  E12000007  London       985       985   
4  E09000002  Barking and Dagenham  E12000007  London      1703      1699   

   COMESTRES  POPDEN  HHOLDS  AVHHOLDSZ  \
0          0 

In [9]:
# 保留关键字段
lsoa = lsoa[["LSOA11CD", "LSOA11NM", "LAD11NM", "geometry"]].copy()

# 先保存一个英国投影版本（面积计算、点转centroid更稳）
lsoa_bng = lsoa.to_crs(epsg=27700)

# 再转成经纬度版本（给 OSM 用）
lsoa_wgs = lsoa.to_crs(epsg=4326)

print(lsoa_bng.head())
print(len(lsoa_bng))
print(lsoa_bng.crs, lsoa_wgs.crs)

    LSOA11CD                   LSOA11NM               LAD11NM  \
0  E01000001        City of London 001A        City of London   
1  E01000002        City of London 001B        City of London   
2  E01000003        City of London 001C        City of London   
3  E01000005        City of London 001E        City of London   
4  E01000006  Barking and Dagenham 016A  Barking and Dagenham   

                                            geometry  
0  POLYGON ((532105.092 182011.23, 532162.491 181...  
1  POLYGON ((532746.813 181786.891, 532671.688 18...  
2  POLYGON ((532135.145 182198.119, 532158.25 182...  
3  POLYGON ((533807.946 180767.77, 533649.063 180...  
4  POLYGON ((545122.049 184314.931, 545271.917 18...  
4835
EPSG:27700 EPSG:4326


In [10]:
print(lsoa.columns)

Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'geometry'], dtype='object')


In [11]:
london_polygon = lsoa_wgs.unary_union
print(london_polygon.geom_type)

/tmp/ipykernel_53626/2800156821.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  london_polygon = lsoa_wgs.unary_union


MultiPolygon


In [12]:
tags_total = {
    "amenity": True,
    "shop": True,
    "leisure": True,
    "tourism": True
}

tags_retail = {
    "shop": True,
    "amenity": ["restaurant", "cafe", "fast_food"]
}

tags_night = {
    "amenity": ["bar", "pub", "nightclub"]
}

In [13]:
def fetch_poi(polygon, tags, poi_name="poi"):
    print(f"Fetching {poi_name} ...")
    gdf = ox.features_from_polygon(polygon, tags)
    print(f"{poi_name} raw rows:", len(gdf))

    # 去掉空几何
    gdf = gdf[gdf.geometry.notnull()].copy()

    # 转到英国投影，centroid 更稳
    gdf = gdf.to_crs(epsg=27700)

    # 所有几何统一转成点
    gdf["geometry"] = gdf.geometry.centroid

    # 保留一份简洁字段
    gdf = gdf.reset_index()

    # 标记类型
    gdf["poi_type"] = poi_name

    print(f"{poi_name} cleaned rows:", len(gdf))
    return gdf

In [ ]:
#####nightpoi点抓取、空间连接到lsoa、合并lsoa上的点、算density

In [14]:
pois_night = fetch_poi(london_polygon, tags_night, poi_name="night")
pois_night.head()

Fetching night ...
night raw rows: 4537
night cleaned rows: 4537


,element,id,geometry,addr:city,addr:housename,addr:housenumber,addr:postcode,addr:street,amenity,cuisine,...,internet_access:wlan:key,drive_through,hotel,old_name2,man_made,check_date:website,note:addr:housenumber,addr:unit:ref,kids_area,poi_type
0,node,451152,POINT (525134.35 190654.283),London,King of Prussia,363,N3 1DH,Regents Park Road,pub,pizza;burger,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,night
1,node,451154,POINT (525039.458 190511.556),NaN,NaN,319,N3 1DP,Regents Park Road,pub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,night
2,node,451271,POINT (526347.745 192160.305),London,NaN,749,N12 0BP,High Road,pub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,night
3,node,12242503,POINT (540575.076 190077.357),NaN,NaN,NaN,NaN,NaN,pub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,night
4,node,15262028,POINT (527085.333 181315.991),London,NaN,30,W2 1JQ,Southwick Street,pub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,night


In [15]:
def aggregate_poi_to_lsoa(pois_gdf, lsoa_gdf, lsoa_id="LSOA11CD", count_col="poi_count"):
    # 保证投影一致
    pois = pois_gdf.to_crs(lsoa_gdf.crs).copy()
    lsoa = lsoa_gdf.copy()

    # 空间连接
    joined = gpd.sjoin(pois, lsoa[[lsoa_id, "geometry"]], how="inner", predicate="within")

    # 统计每个 LSOA 的 POI 数量
    counts = joined.groupby(lsoa_id).size().reset_index(name=count_col)

    # 合并回 LSOA
    out = lsoa.merge(counts, on=lsoa_id, how="left")
    out[count_col] = out[count_col].fillna(0)

    return out, joined

In [16]:
lsoa_night, joined_night = aggregate_poi_to_lsoa(
    pois_night,
    lsoa_bng,
    lsoa_id="LSOA11CD",
    count_col="night_poi_count"
)

print(lsoa_night[["LSOA11CD", "night_poi_count"]].head())

    LSOA11CD  night_poi_count
0  E01000001              4.0
1  E01000002              5.0
2  E01000003              1.0
3  E01000005             19.0
4  E01000006              0.0


In [17]:
# 面积（km²）
lsoa_night["area_km2"] = lsoa_night.geometry.area / 1_000_000

# density
lsoa_night["night_poi_density"] = lsoa_night["night_poi_count"] / lsoa_night["area_km2"]

lsoa_night[["LSOA11CD", "night_poi_count", "area_km2", "night_poi_density"]].head()

,LSOA11CD,night_poi_count,area_km2,night_poi_density
0,E01000001,4.0,0.133321,30.002827
1,E01000002,5.0,0.226191,22.105185
2,E01000003,1.0,0.057303,17.451104
3,E01000005,19.0,0.190739,99.612685
4,E01000006,0.0,0.144196,0.000000


In [18]:
# 1) 保存带空间的版本
lsoa_night.to_file("dataset/lsoa_night.gpkg", driver="GPKG")

# 2) 保存纯表格版本
lsoa_night.drop(columns="geometry").to_csv("dataset/lsoa_night.csv", index=False)

print("night 已保存")

night 已保存


In [19]:
for f in os.listdir("dataset"):
    if "night" in f:
        print(f)

lsoa_night.gpkg
lsoa_night.csv


In [20]:
del lsoa_night
del pois_night
del lsoa
del lsoa_bng
del lsoa_wgs
del london_polygon

import gc
gc.collect()

print("内存已清理")

内存已清理


In [21]:
del joined_night
gc.collect()

0

In [22]:
import geopandas as gpd
import osmnx as ox
import gc

In [23]:
# 重新读 LSOA
lsoa = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp", engine="pyogrio")
lsoa = lsoa[["LSOA11CD", "LSOA11NM", "LAD11NM", "geometry"]].copy()

In [24]:
lsoa_bng = lsoa.to_crs(epsg=27700)
lsoa_wgs = lsoa.to_crs(epsg=4326)
london_polygon = lsoa_wgs.union_all()

In [25]:
tags_retail = {
    "shop": True,
    "amenity": ["restaurant", "cafe", "fast_food"]
}

In [26]:
def fetch_poi_light(polygon, tags):
    gdf = ox.features_from_polygon(polygon, tags)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf.to_crs(epsg=27700)
    gdf["geometry"] = gdf.geometry.centroid
    gdf = gdf[["geometry"]].copy()
    return gdf

In [27]:
def aggregate_poi_to_lsoa_light(pois_gdf, lsoa_gdf, lsoa_id, count_col):
    joined = gpd.sjoin(
        pois_gdf,
        lsoa_gdf[[lsoa_id, "geometry"]],
        how="inner",
        predicate="within"
    )
    counts = joined.groupby(lsoa_id).size().reset_index(name=count_col)
    out = lsoa_gdf.merge(counts, on=lsoa_id, how="left")
    out[count_col] = out[count_col].fillna(0)
    del joined, counts
    gc.collect()
    return out

In [29]:
pois_retail = fetch_poi_light(london_polygon, tags_retail)

KeyboardInterrupt: 

In [30]:
tags_food = {
    "amenity": ["restaurant", "cafe", "fast_food"]
}


In [31]:
pois_food = fetch_poi_light(london_polygon, tags_food)
print("food POIs:", len(pois_food))

food POIs: 21802


In [32]:
pois_food.head()

geometry
element id                                     
node    451153    POINT (525207.599 190788.532)
        20849687    POINT (515960.4 169314.811)
        25475389  POINT (529871.799 182510.232)
        25497832   POINT (530788.32 182331.603)
        25744422  POINT (528969.807 186759.799)

In [ ]:
#####foodpoi点抓取，合并，density计算

In [33]:
lsoa_food = aggregate_poi_to_lsoa_light(
    pois_food,
    lsoa_bng,
    lsoa_id="LSOA11CD",
    count_col="food_poi_count"
)

lsoa_food["area_km2"] = lsoa_food.geometry.area / 1_000_000
lsoa_food["food_poi_density"] = lsoa_food["food_poi_count"] / lsoa_food["area_km2"]

print(lsoa_food[["food_poi_count", "food_poi_density"]].describe())

       food_poi_count  food_poi_density
count     4835.000000       4835.000000
mean         4.508790         23.890635
std         14.397041         55.779096
min          0.000000          0.000000
25%          0.000000          0.000000
50%          1.000000          3.691477
75%          4.000000         22.448862
max        553.000000       1002.161471


In [34]:
lsoa_food.sort_values("food_poi_count", ascending=False)[
    ["LSOA11CD", "LSOA11NM", "LAD11NM", "food_poi_count", "area_km2", "food_poi_density"]
].head(10)

,LSOA11CD,LSOA11NM,LAD11NM,food_poi_count,area_km2,food_poi_density
4674,E01032739,City of London 001F,City of London,553.0,1.658496,333.434585
4613,E01004734,Westminster 018A,Westminster,321.0,0.320308,1002.161471
4782,E01033595,Westminster 013E,Westminster,254.0,0.515400,492.821458
4640,E01004763,Westminster 013B,Westminster,224.0,0.294390,760.896189
4614,E01004735,Westminster 018B,Westminster,161.0,0.487142,330.499437
4615,E01004736,Westminster 018C,Westminster,143.0,1.192057,119.960666
4806,E01033708,Hackney 027G,Hackney,133.0,0.355770,373.837553
4675,E01032740,City of London 001G,City of London,122.0,0.639349,190.819005
4770,E01033583,Newham 013G,Newham,122.0,1.584996,76.971824
4199,E01004307,Tower Hamlets 015B,Tower Hamlets,110.0,0.211750,519.481738


In [35]:
lsoa_food.sort_values("food_poi_density", ascending=False)[
    ["LSOA11CD", "LSOA11NM", "LAD11NM", "food_poi_count", "area_km2", "food_poi_density"]
].head(10)


,LSOA11CD,LSOA11NM,LAD11NM,food_poi_count,area_km2,food_poi_density
4613,E01004734,Westminster 018A,Westminster,321.0,0.320308,1002.161471
4783,E01033596,Westminster 013F,Westminster,99.0,0.107507,920.868464
4640,E01004763,Westminster 013B,Westminster,224.0,0.294390,760.896189
901,E01000919,Camden 028D,Camden,94.0,0.149934,626.940816
3489,E01003566,Newham 014D,Newham,59.0,0.099099,595.366487
4199,E01004307,Tower Hamlets 015B,Tower Hamlets,110.0,0.211750,519.481738
4414,E01004525,Wandsworth 035A,Wandsworth,70.0,0.139886,500.406500
835,E01000853,Camden 025A,Camden,29.0,0.058451,496.143701
833,E01000851,Camden 026B,Camden,67.0,0.135560,494.244965
4782,E01033595,Westminster 013E,Westminster,254.0,0.515400,492.821458


In [36]:
lsoa_food["area_km2"].describe()

count    4835.000000
mean        0.325441
std         0.629009
min         0.016902
25%         0.133548
50%         0.203355
75%         0.319056
max        15.808727
Name: area_km2, dtype: float64

In [37]:
lsoa_food.drop(columns="geometry").to_csv(
    "dataset/lsoa_food.csv",
    index=False
)

print("CSV saved ✅")

CSV saved ✅


In [38]:
lsoa_food.to_file(
    "dataset/lsoa_food.gpkg",
    driver="GPKG"
)

print("GPKG saved ✅")

GPKG saved ✅


In [39]:
lsoa_bng.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [40]:
# 删除 food 相关变量
del pois_food
del lsoa_food

# 如果你有这些，也一起删
try:
    del joined_food
except:
    pass

# 强制垃圾回收
import gc
gc.collect()

print("food 内存已清理 ✅")

food 内存已清理 ✅


In [41]:
print([var for var in globals().keys() if not var.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'json', 'sys', 'islice', 'collections', 'NamespaceMagics', 'ox', 'gpd', 'pd', 'os', 'tags_total', 'tags_retail', 'tags_night', 'fetch_poi', 'aggregate_poi_to_lsoa', 'f', 'gc', 'lsoa', 'lsoa_bng', 'lsoa_wgs', 'london_polygon', 'fetch_poi_light', 'aggregate_poi_to_lsoa_light', 'tags_food']


In [42]:
import geopandas as gpd
import pandas as pd
import os
import gc
import osmnx as ox

lsoa = gpd.read_file(
    "dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp",
    engine="pyogrio"
)
lsoa = lsoa[["LSOA11CD", "LSOA11NM", "LAD11NM", "geometry"]].copy()

lsoa_bng = lsoa.to_crs(epsg=27700)
lsoa_wgs = lsoa.to_crs(epsg=4326)

In [43]:
os.makedirs("dataset/shop_chunks", exist_ok=True)

In [ ]:
#####shop分快读取、合并、连接

In [44]:
def fetch_poi_light(polygon, tags):
    gdf = ox.features_from_polygon(polygon, tags)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf.to_crs(epsg=27700)
    gdf["geometry"] = gdf.geometry.centroid
    gdf = gdf.reset_index()[["element", "id", "geometry"]].drop_duplicates(subset=["element", "id"])
    return gdf

In [45]:
boroughs = sorted(lsoa_wgs["LAD11NM"].unique().tolist())
print("borough count:", len(boroughs))
print(boroughs)

borough count: 33
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'City of London', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster']


In [46]:
shop_files = []

for b in boroughs:
    safe_name = (
        b.replace("/", "_")
         .replace(" ", "_")
         .replace(",", "")
         .replace("'", "")
    )
    out_path = f"dataset/shop_chunks/shop_{safe_name}.gpkg"

    # 如果已经抓过，就跳过，支持断点续跑
    if os.path.exists(out_path):
        print(f"skip existing: {b}")
        shop_files.append(out_path)
        continue

    print(f"Fetching shops in: {b}")

    sub = lsoa_wgs[lsoa_wgs["LAD11NM"] == b]
    poly = sub.union_all()

    try:
        pois_shop_b = fetch_poi_light(poly, {"shop": True})
        print(f"  rows: {len(pois_shop_b)}")

        pois_shop_b.to_file(out_path, driver="GPKG")
        shop_files.append(out_path)

        del pois_shop_b
        gc.collect()

    except Exception as e:
        print(f"  failed: {b} -> {e}")
        gc.collect()

Fetching shops in: Barking and Dagenham
  rows: 512
Fetching shops in: Barnet
  rows: 1153
Fetching shops in: Bexley
  rows: 824
Fetching shops in: Brent
  rows: 1076
Fetching shops in: Bromley
  rows: 1446
Fetching shops in: Camden
  rows: 2158
Fetching shops in: City of London
  rows: 586
Fetching shops in: Croydon
  rows: 1473
Fetching shops in: Ealing
  rows: 1187
Fetching shops in: Enfield
  rows: 1027
Fetching shops in: Greenwich
  rows: 1004
Fetching shops in: Hackney
  rows: 1287
Fetching shops in: Hammersmith and Fulham
  rows: 1354
Fetching shops in: Haringey
  rows: 1518
Fetching shops in: Harrow
  rows: 877
Fetching shops in: Havering
  rows: 834
Fetching shops in: Hillingdon
  rows: 962
Fetching shops in: Hounslow
  rows: 931
Fetching shops in: Islington
  rows: 1775
Fetching shops in: Kensington and Chelsea
  rows: 1957
Fetching shops in: Kingston upon Thames
  rows: 568
Fetching shops in: Lambeth
  rows: 1450
Fetching shops in: Lewisham
  rows: 1295
Fetching shops in: Me

In [47]:
b = "Waltham Forest"

safe_name = (
    b.replace("/", "_")
     .replace(" ", "_")
     .replace(",", "")
     .replace("'", "")
)

out_path = f"dataset/shop_chunks/shop_{safe_name}.gpkg"

sub = lsoa_wgs[lsoa_wgs["LAD11NM"] == b]
poly = sub.union_all()

pois_shop_b = fetch_poi_light(poly, {"shop": True})
print("rows:", len(pois_shop_b))

pois_shop_b.to_file(out_path, driver="GPKG")
print("saved:", out_path)

rows: 1673
saved: dataset/shop_chunks/shop_Waltham_Forest.gpkg


In [48]:
import os
import pandas as pd
import geopandas as gpd

shop_files = sorted([
    os.path.join("dataset/shop_chunks", f)
    for f in os.listdir("dataset/shop_chunks")
    if f.endswith(".gpkg")
])

print("saved borough files:", len(shop_files))

shop_gdfs = [gpd.read_file(f) for f in shop_files]
pois_shop = pd.concat(shop_gdfs, ignore_index=True)

pois_shop = pois_shop.drop_duplicates(subset=["element", "id"]).copy()
pois_shop = gpd.GeoDataFrame(pois_shop, geometry="geometry", crs="EPSG:27700")

print("total unique shop POIs:", len(pois_shop))

saved borough files: 33
total unique shop POIs: 42868


In [49]:
lsoa_shop = aggregate_poi_to_lsoa_light(
    pois_shop,
    lsoa_bng,
    lsoa_id="LSOA11CD",
    count_col="shop_poi_count"
)

lsoa_shop["area_km2"] = lsoa_shop.geometry.area / 1_000_000
lsoa_shop["shop_poi_density"] = lsoa_shop["shop_poi_count"] / lsoa_shop["area_km2"]

print(lsoa_shop[["shop_poi_count", "shop_poi_density"]].describe())

       shop_poi_count  shop_poi_density
count     4835.000000       4835.000000
mean         8.865357         48.106087
std         21.499340         99.806246
min          0.000000          0.000000
25%          0.000000          0.000000
50%          2.000000          9.766177
75%          9.000000         48.444523
max        606.000000       1460.367160


In [50]:
lsoa_shop.to_file("dataset/lsoa_shop.gpkg", driver="GPKG")
lsoa_shop.drop(columns="geometry").to_csv("dataset/lsoa_shop.csv", index=False)

print("shop 已保存")

shop 已保存


In [ ]:
#获取population density数据第二个需要读取本地数据

In [54]:
del lsoa_shop

In [55]:
import pandas as pd

In [59]:
pop = pd.read_csv("dataset/population/census2021-ts001-lsoa.csv")

In [62]:
# 选你需要的列
pop = pop[[
    "geography code",
    "Residence type: Total; measures: Value"
]]

In [66]:
# 重命名（非常重要，后面好用）
pop = pop.rename(columns={
    "geography code": "LSOA_code",
    "Residence type: Total; measures: Value": "population"
})

pop.head()

,LSOA_code,population
0,E01011954,2284
1,E01011969,1344
2,E01011970,1070
3,E01011971,1323
4,E01033465,1955


In [67]:
pop = pop[pop["LSOA_code"].isin(lsoa["LSOA11CD"])]

In [68]:
print(len(pop))
print(len(lsoa))

4659
4835


In [69]:
pop["LSOA_code"].head(20)

19783    E01000001
19784    E01000002
19785    E01000003
19786    E01000005
19787    E01032739
19788    E01032740
19789    E01000027
19790    E01000028
19791    E01000029
19792    E01000030
19793    E01000031
19794    E01000032
19795    E01000110
19796    E01000111
19797    E01000112
19798    E01000113
19799    E01000034
19800    E01000037
19801    E01000038
19802    E01000039
Name: LSOA_code, dtype: object

In [70]:
# merge
lsoa_pop = lsoa.merge(pop, left_on="LSOA11CD", right_on="LSOA_code", how="left")

In [71]:
# 检查
print("Total LSOA:", len(lsoa_pop))
print("Missing population:", lsoa_pop["population"].isna().sum())

# 看几条缺失的
print(lsoa_pop.loc[lsoa_pop["population"].isna(), ["LSOA11CD"]].head(20))

Total LSOA: 4835
Missing population: 176
      LSOA11CD
8    E01000010
21   E01000023
44   E01000048
88   E01000092
105  E01000109
121  E01000125
144  E01000148
150  E01000155
257  E01000262
373  E01000378
426  E01000432
476  E01000482
577  E01000588
587  E01000600
621  E01000635
660  E01000675
725  E01000740
834  E01000852
836  E01000854
846  E01000864


In [72]:
lsoa_pop.loc[lsoa_pop["population"].isna(), "LSOA11CD"].head()

8      E01000010
21     E01000023
44     E01000048
88     E01000092
105    E01000109
Name: LSOA11CD, dtype: object

In [73]:
lsoa_pop = lsoa_pop.dropna(subset=["population"]).copy()
print("After dropping missing:", len(lsoa_pop))

After dropping missing: 4659


In [74]:
# 投影到米制坐标系
lsoa_proj = lsoa_pop.to_crs(epsg=27700)

In [75]:
# 面积（km²）
lsoa_pop["area_km2"] = lsoa_proj.geometry.area / 1e6

In [76]:

# 人口密度
lsoa_pop["pop_density"] = lsoa_pop["population"] / lsoa_pop["area_km2"]

In [77]:

# 检查结果
print(lsoa_pop[["LSOA11CD", "population", "area_km2", "pop_density"]].head())
print(lsoa_pop["pop_density"].describe())

    LSOA11CD  population  area_km2   pop_density
0  E01000001      1475.0  0.133321  11063.542564
1  E01000002      1384.0  0.226191   6118.715289
2  E01000003      1613.0  0.057303  28148.629970
3  E01000005      1100.0  0.190739   5767.050167
4  E01000006      1845.0  0.144196  12795.098065
count     4659.000000
mean      9965.405223
std       6112.419708
min        119.807245
25%       5462.502089
50%       8819.427281
75%      13378.054115
max      60053.082507
Name: pop_density, dtype: float64


In [78]:
lsoa_pop.to_file("lsoa_with_pop.geojson", driver="GeoJSON")

In [80]:
df = lsoa_pop.drop(columns="geometry")
df.to_csv("lsoa_pop_raw.csv", index=False)

In [ ]:
#计算workplace density第三个读取本地数据

In [81]:
wp = pd.read_csv("dataset/workplacepopulation/WP001_lsoa.csv")
wp.head()

,Lower layer Super Output Areas Code,Lower layer Super Output Areas Label,Count
0,E01000001,City of London 001A,3256
1,E01000002,City of London 001B,7230
2,E01000003,City of London 001C,864
3,E01000005,City of London 001E,3441
4,E01000006,Barking and Dagenham 016A,422


In [82]:
print(wp.columns)

Index(['Lower layer Super Output Areas Code',
       'Lower layer Super Output Areas Label', 'Count'],
      dtype='object')


In [85]:
wp = wp[[
    "Lower layer Super Output Areas Code",
    "Count"
]]

In [86]:
wp = wp.rename(columns={
    "Lower layer Super Output Areas Code": "LSOA_code",
    "Count": "workplace_pop"
})

In [87]:
wp = wp[wp["LSOA_code"].isin(lsoa_pop["LSOA11CD"])]

In [88]:
print(len(wp))

4659


In [89]:
lsoa_pop = lsoa_pop.merge(
    wp,
    left_on="LSOA11CD",
    right_on="LSOA_code",
    how="left"
)

In [90]:
print("Missing workplace:", lsoa_pop["workplace_pop"].isna().sum())

Missing workplace: 0


In [91]:
lsoa_pop["workplace_density"] = lsoa_pop["workplace_pop"] / lsoa_pop["area_km2"]

In [92]:
lsoa_pop["workplace_density"].describe()

count     4659.000000
mean      4539.966521
std       4511.346893
min         76.073967
25%       1983.658144
50%       3297.459798
75%       5744.375598
max      83923.465119
Name: workplace_density, dtype: float64

In [94]:
lsoa_pop.to_file("lsoa_workplacepopulation.geojson", driver="GeoJSON")

In [95]:
df = lsoa_pop.drop(columns="geometry")
df.to_csv("lsoa_workplacepopulation.csv", index=False)

In [ ]:
#intersection density第四个本地数据

In [4]:
import geopandas as gpd
import pandas as pd
import glob

In [7]:
# 2. 读所有 RoadNode
roadnode_files = glob.glob("dataset/oproad_essh_gb/data/**/*RoadNode.shp", recursive=True)

gdfs = []
for f in roadnode_files:
    g = gpd.read_file(f)[["formOfNode", "geometry"]]
    gdfs.append(g)

roadnode = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

In [8]:
# 1. 读 LSOA
lsoa = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp")
lsoa["area_km2"] = lsoa.geometry.area / 1e6
lsoa = lsoa[["LSOA11CD", "area_km2", "geometry"]].copy()

In [9]:
# 3. 只保留 junction + roundabout
roadnode_intersections = roadnode[
    roadnode["formOfNode"].isin(["junction", "roundabout"])
][["geometry"]].copy()

print(len(roadnode_intersections))

2136197


In [10]:
import numpy as np
import gc

In [13]:
lsoa = lsoa.to_crs(epsg=27700)
roadnode_intersections = roadnode_intersections.to_crs(epsg=27700)

print(lsoa.crs)
print(roadnode_intersections.crs)

EPSG:27700
EPSG:27700


In [14]:
chunk_size = 200000   # 先从20万试起，还是崩就改成100000
results = []

In [16]:
for start in range(0, len(roadnode_intersections), chunk_size):
    end = min(start + chunk_size, len(roadnode_intersections))
    chunk = roadnode_intersections.iloc[start:end].copy()
    
    joined_chunk = gpd.sjoin(
        chunk,
        lsoa,
        how="left",
        predicate="within"
    )

    count_chunk = (
        joined_chunk.groupby("LSOA11CD")
        .size()
        .reset_index(name="n")
    )
    results.append(count_chunk)

    print(f"done: {start} - {end}")

    del chunk, joined_chunk, count_chunk
    gc.collect()

done: 0 - 200000
done: 200000 - 400000
done: 400000 - 600000
done: 600000 - 800000
done: 800000 - 1000000
done: 1000000 - 1200000
done: 1200000 - 1400000
done: 1400000 - 1600000
done: 1600000 - 1800000
done: 1800000 - 2000000
done: 2000000 - 2136197


In [17]:
intersection_count = (
    pd.concat(results, ignore_index=True)
    .groupby("LSOA11CD", as_index=False)["n"]
    .sum()
    .rename(columns={"n": "intersection_count"})
)

print(intersection_count.head())
print(len(intersection_count))

    LSOA11CD  intersection_count
0  E01000001                  23
1  E01000002                  26
2  E01000003                   5
3  E01000005                  39
4  E01000006                  10
4832


In [18]:
lsoa = lsoa.merge(intersection_count, on="LSOA11CD", how="left")
lsoa["intersection_count"] = lsoa["intersection_count"].fillna(0)

lsoa["intersection_density"] = (
    lsoa["intersection_count"] / lsoa["area_km2"]
)

print(lsoa["intersection_density"].describe())

count    4835.000000
mean      107.626184
std        53.674669
min         0.000000
25%        70.170593
50%       100.377787
75%       137.996248
max       464.446170
Name: intersection_density, dtype: float64


In [19]:
lsoa.to_file("lsoa_with_intersection_density.geojson", driver="GeoJSON")
lsoa.drop(columns="geometry").to_csv("lsoa_with_intersection_density.csv", index=False)

In [ ]:
#路网 可达性变量

In [40]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import networkx as nx
import gc
import osmnx as ox
from shapely.validation import make_valid

In [41]:
# 读取 LSOA 边界
lsoa = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp")

# 只保留需要的列
lsoa = lsoa[["LSOA11CD", "geometry"]].copy()


In [42]:
# 面积（km²）
lsoa["area_km2"] = lsoa.geometry.area / 1e6

print(lsoa.shape)
print(lsoa.crs)
lsoa.head()

(4835, 3)
PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


,LSOA11CD,geometry,area_km2
0,E01000001,"POLYGON ((532105.092 182011.23, 532162.491 181...",0.133321
1,E01000002,"POLYGON ((532746.813 181786.891, 532671.688 18...",0.226191
2,E01000003,"POLYGON ((532135.145 182198.119, 532158.25 182...",0.057303
3,E01000005,"POLYGON ((533807.946 180767.77, 533649.063 180...",0.190739
4,E01000006,"POLYGON ((545122.049 184314.931, 545271.917 18...",0.144196


In [19]:
#####LMD_sore第五个本地数据

In [44]:
import pandas as pd
import geopandas as gpd

In [45]:
lsoa = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp")

# 只保留需要的列
lsoa = lsoa[["LSOA11CD", "geometry"]].copy()

print("LSOA rows:", len(lsoa))
print(lsoa.head())


LSOA rows: 4835
    LSOA11CD                                           geometry
0  E01000001  POLYGON ((532105.092 182011.23, 532162.491 181...
1  E01000002  POLYGON ((532746.813 181786.891, 532671.688 18...
2  E01000003  POLYGON ((532135.145 182198.119, 532158.25 182...
3  E01000005  POLYGON ((533807.946 180767.77, 533649.063 180...
4  E01000006  POLYGON ((545122.049 184314.931, 545271.917 18...


In [46]:
imd = pd.read_csv("dataset/IMD2019_Deprivation.csv")

In [47]:
print("IMD columns:")
for col in imd.columns:
    print(col)

print(imd.head())

IMD columns:
LSOA code (2011)
LSOA name (2011)
Local Authority District code (2019)
Local Authority District name (2019)
Index of Multiple Deprivation (IMD) Rank
Index of Multiple Deprivation (IMD) Decile
  LSOA code (2011)           LSOA name (2011)  \
0        E01000001        City of London 001A   
1        E01000002        City of London 001B   
2        E01000003        City of London 001C   
3        E01000005        City of London 001E   
4        E01000006  Barking and Dagenham 016A   

  Local Authority District code (2019) Local Authority District name (2019)  \
0                            E09000001                       City of London   
1                            E09000001                       City of London   
2                            E09000001                       City of London   
3                            E09000001                       City of London   
4                            E09000002                 Barking and Dagenham   

  Index of Multiple Depri

In [ ]:
#####算betweenness

In [53]:
# 1. 重命名
imd = imd.rename(columns={
    "LSOA code (2011)": "LSOA_code",
    "Index of Multiple Deprivation (IMD) Rank": "imd_rank"
})

In [54]:

# 2. 去掉逗号，并转成数值
imd["imd_rank"] = (
    imd["imd_rank"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

imd["imd_rank"] = pd.to_numeric(imd["imd_rank"], errors="coerce")

In [55]:
# 3. 检查转换是否成功
print(imd["imd_rank"].dtype)
print(imd["imd_rank"].head())
print(imd["imd_rank"].isna().sum())

int64
0    29199
1    30379
2    14915
3     8678
4    14486
Name: imd_rank, dtype: int64
0


In [57]:
# 转 score
imd["imd_score"] = imd["imd_rank"].max() - imd["imd_rank"]

In [58]:
# 5. 保留需要的列
imd = imd[["LSOA_code", "imd_score"]].copy()

In [59]:
# 6. 筛 London
imd = imd[imd["LSOA_code"].isin(lsoa["LSOA11CD"])].copy()

In [60]:
# 7. merge
lsoa = lsoa.merge(imd, left_on="LSOA11CD", right_on="LSOA_code", how="left")

In [61]:
# 8. 检查
print(lsoa["imd_score"].describe())
print("missing imd:", lsoa["imd_score"].isna().sum())
print(lsoa[["LSOA11CD", "imd_score"]].head())

count     4835.000000
mean     17646.471975
std       8093.221202
min        281.000000
25%      11176.500000
50%      18811.000000
75%      24642.000000
max      32298.000000
Name: imd_score, dtype: float64
missing imd: 0
    LSOA11CD  imd_score
0  E01000001       3645
1  E01000002       2465
2  E01000003      17929
3  E01000005      24166
4  E01000006      18358


In [62]:
imd.to_csv("dataset/imd_lsoa.csv", index=False)

In [63]:
lsoa.to_file("dataset/lsoa_with_imd.gpkg", driver="GPKG")

In [64]:
#尝试算between

In [1]:
import os
import gc
import glob
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import box

In [2]:
os.makedirs("dataset/network_chunk", exist_ok=True)
os.makedirs("dataset/network_node", exist_ok=True)

In [3]:
lsoa = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp")
lsoa = lsoa[["LSOA11CD", "geometry"]].copy()
lsoa_wgs = lsoa.to_crs(epsg=4326)

In [4]:
def make_grid(bounds, n_cols=6, n_rows=6):
    minx, miny, maxx, maxy = bounds
    dx = (maxx - minx) / n_cols
    dy = (maxy - miny) / n_rows
    cells = []
    for i in range(n_cols):
        for j in range(n_rows):
            x1 = minx + i * dx
            x2 = x1 + dx
            y1 = miny + j * dy
            y2 = y1 + dy
            cells.append(box(x1, y1, x2, y2))
    return gpd.GeoDataFrame(geometry=cells, crs="EPSG:4326")

In [5]:
grid = make_grid(lsoa_wgs.total_bounds, n_cols=6, n_rows=6)

In [6]:
london_union = lsoa_wgs.geometry.union_all()
grid = grid[grid.intersects(london_union)].copy().reset_index(drop=True)

print("有效网格数:", len(grid))

有效网格数: 33


In [7]:
ox.settings.use_cache = True
ox.settings.log_console = True

In [8]:
for i, row in grid.iterrows():
    out_graph = f"dataset/network_chunks/chunk_{i:02d}.graphml"
    if os.path.exists(out_graph):
        print("已存在，跳过:", out_graph)
        continue

    print(f"抓取第 {i+1}/{len(grid)} 块")
    try:
        poly = row.geometry   # 先不要 buffer，减小范围
        G_part = ox.graph_from_polygon(
            poly,
            network_type="drive",
            simplify=False,
            retain_all=False,
            truncate_by_edge=True
        )
        ox.save_graphml(G_part, out_graph)
        print("保存成功:", out_graph)

        del G_part
        gc.collect()

    except Exception as e:
        print(f"第 {i+1} 块失败: {e}")
        gc.collect()

抓取第 1/33 块
保存成功: dataset/network_chunks/chunk_00.graphml
抓取第 2/33 块
保存成功: dataset/network_chunks/chunk_01.graphml
抓取第 3/33 块
保存成功: dataset/network_chunks/chunk_02.graphml
抓取第 4/33 块
保存成功: dataset/network_chunks/chunk_03.graphml
抓取第 5/33 块
保存成功: dataset/network_chunks/chunk_04.graphml
抓取第 6/33 块
保存成功: dataset/network_chunks/chunk_05.graphml
抓取第 7/33 块
保存成功: dataset/network_chunks/chunk_06.graphml
抓取第 8/33 块
保存成功: dataset/network_chunks/chunk_07.graphml
抓取第 9/33 块
保存成功: dataset/network_chunks/chunk_08.graphml
抓取第 10/33 块
保存成功: dataset/network_chunks/chunk_09.graphml
抓取第 11/33 块
保存成功: dataset/network_chunks/chunk_10.graphml
抓取第 12/33 块
保存成功: dataset/network_chunks/chunk_11.graphml
抓取第 13/33 块
保存成功: dataset/network_chunks/chunk_12.graphml
抓取第 14/33 块
保存成功: dataset/network_chunks/chunk_13.graphml
抓取第 15/33 块
保存成功: dataset/network_chunks/chunk_14.graphml
抓取第 16/33 块
保存成功: dataset/network_chunks/chunk_15.graphml
抓取第 17/33 块
保存成功: dataset/network_chunks/chunk_16.graphml
抓取第 18/33 块
保存成功: datas

In [9]:
graph_files = sorted(glob.glob("dataset/network_chunks/*.graphml"))

In [10]:
for graph_path in graph_files:
    name = os.path.splitext(os.path.basename(graph_path))[0]
    out_nodes = f"dataset/network_nodes/{name}_betweenness.gpkg"

    if os.path.exists(out_nodes):
        print("已存在，跳过:", out_nodes)
        continue

    print("处理:", graph_path)
    try:
        G = ox.load_graphml(graph_path)

        # 投影到米制
        G = ox.project_graph(G, to_crs="EPSG:27700")

        # 转无向图
        G_u = ox.convert.to_undirected(G)

        # approximate betweenness
        bet = nx.betweenness_centrality(
            G_u,
            k=100,               # 先保守一点
            normalized=True,
            weight="length",
            endpoints=False,
            seed=42
        )

        nodes, edges = ox.graph_to_gdfs(G_u, nodes=True, edges=True)
        nodes["betweenness"] = nodes.index.map(bet)
        nodes = nodes[["betweenness", "geometry"]].copy()

        nodes.to_file(out_nodes, driver="GPKG")
        print("保存成功:", out_nodes)

        del G, G_u, bet, nodes, edges
        gc.collect()

    except Exception as e:
        print("失败:", graph_path, e)
        gc.collect()

处理: dataset/network_chunks/chunk_00.graphml
保存成功: dataset/network_nodes/chunk_00_betweenness.gpkg
处理: dataset/network_chunks/chunk_01.graphml
保存成功: dataset/network_nodes/chunk_01_betweenness.gpkg
处理: dataset/network_chunks/chunk_02.graphml
保存成功: dataset/network_nodes/chunk_02_betweenness.gpkg
处理: dataset/network_chunks/chunk_03.graphml
保存成功: dataset/network_nodes/chunk_03_betweenness.gpkg
处理: dataset/network_chunks/chunk_04.graphml
保存成功: dataset/network_nodes/chunk_04_betweenness.gpkg
处理: dataset/network_chunks/chunk_05.graphml
保存成功: dataset/network_nodes/chunk_05_betweenness.gpkg
处理: dataset/network_chunks/chunk_06.graphml
保存成功: dataset/network_nodes/chunk_06_betweenness.gpkg
处理: dataset/network_chunks/chunk_07.graphml
保存成功: dataset/network_nodes/chunk_07_betweenness.gpkg
处理: dataset/network_chunks/chunk_08.graphml
保存成功: dataset/network_nodes/chunk_08_betweenness.gpkg
处理: dataset/network_chunks/chunk_09.graphml
保存成功: dataset/network_nodes/chunk_09_betweenness.gpkg
处理: dataset/network_

In [11]:
node_files = sorted(glob.glob("dataset/network_nodes/*_betweenness.gpkg"))
gdfs = [gpd.read_file(f) for f in node_files]

nodes_all = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

print(nodes_all.shape)
print(nodes_all["betweenness"].describe())

(885113, 3)
count    885113.000000
mean          0.010404
std           0.024689
min           0.000000
25%           0.000148
50%           0.000785
75%           0.008928
max           0.640063
Name: betweenness, dtype: float64


In [12]:
# LSOA 投影到 27700
lsoa_proj = gpd.read_file("dataset/London_boundaries/LSOA_2011_London_gen_MHW.shp")
lsoa_proj = lsoa_proj[["LSOA11CD", "geometry"]].copy().to_crs(epsg=27700)

In [13]:
# 如果 nodes_all 太大，也可以分块 sjoin；先试直接 join
joined = gpd.sjoin(
    nodes_all,
    lsoa_proj,
    how="left",
    predicate="within"
)

betweenness_lsoa = (
    joined.groupby("LSOA11CD", as_index=False)["betweenness"]
    .mean()
)

print(betweenness_lsoa.head())
print("有 betweenness 的 LSOA 数:", len(betweenness_lsoa))

    LSOA11CD  betweenness
0  E01000001     0.005040
1  E01000002     0.001856
2  E01000003     0.000692
3  E01000005     0.006598
4  E01000006     0.008735
有 betweenness 的 LSOA 数: 4803


In [14]:
# merge 回 LSOA
lsoa_proj = lsoa_proj.merge(betweenness_lsoa, on="LSOA11CD", how="left")

print(lsoa_proj["betweenness"].describe())
print("missing betweenness:", lsoa_proj["betweenness"].isna().sum())

count    4803.000000
mean        0.007938
std         0.008288
min         0.000000
25%         0.002259
50%         0.005342
75%         0.010825
max         0.088781
Name: betweenness, dtype: float64
missing betweenness: 32


In [15]:
lsoa_proj["betweenness"] = lsoa_proj["betweenness"].fillna(0)

In [16]:
lsoa_proj["betweenness"] = lsoa_proj["betweenness"].fillna(0)

print(lsoa_proj["betweenness"].describe())
print("missing betweenness:", lsoa_proj["betweenness"].isna().sum())

count    4835.000000
mean        0.007886
std         0.008285
min         0.000000
25%         0.002213
50%         0.005312
75%         0.010748
max         0.088781
Name: betweenness, dtype: float64
missing betweenness: 0


In [17]:
lsoa_proj.drop(columns="geometry").to_csv("dataset/lsoa_betweenness.csv", index=False)
lsoa_proj.to_file("dataset/lsoa_betweenness.gpkg", driver="GPKG")

In [41]:
#####读取crime rate

In [1]:
import pandas as pd

In [2]:
# 读取两个文件
crime_hist = pd.read_csv("dataset/MPS LSOA Level Crime (Historical).csv")
crime_recent = pd.read_csv("dataset/MPS LSOA Level Crime (most recent 24 months).csv")
print("Historical shape:", crime_hist.shape)
print("Recent shape:", crime_recent.shape)

Historical shape: (127288, 173)
Recent shape: (105646, 30)


In [3]:
# 看看列名，先确认一致
print("\nHistorical columns:")
print(crime_hist.columns.tolist())

print("\nRecent columns:")
print(crime_recent.columns.tolist())


Historical columns:
['LSOA Code', 'LSOA Name', 'Borough', 'Major Category', 'Minor Category', '201903', '201904', '201905', '201906', '201907', '201908', '201909', '201910', '201911', '201912', '202001', '202002', '201901', '201902', '201004', '201005', '201006', '201007', '201008', '201009', '201010', '201011', '201012', '201101', '201102', '201103', '201104', '201105', '201106', '201107', '201108', '201109', '201110', '201111', '201112', '201201', '201202', '201203', '201204', '201205', '201206', '201207', '201208', '201209', '201210', '201211', '201212', '201301', '201302', '201303', '201304', '201305', '201306', '201307', '201308', '201309', '201310', '201311', '201312', '201401', '201402', '201403', '201404', '201405', '201406', '201407', '201408', '201409', '201410', '201411', '201412', '201501', '201502', '201503', '201504', '201505', '201506', '201507', '201508', '201509', '201510', '201511', '201512', '201601', '201602', '201603', '201604', '201605', '201606', '201607', '2016

In [4]:
hist_months = [col for col in crime_hist.columns if col.isdigit()]
recent_months = [col for col in crime_recent.columns if col.isdigit()]

In [5]:
overlap = set(hist_months) & set(recent_months)
print("Overlap:", sorted(overlap))

Overlap: ['202403']


In [6]:
crime_hist_clean = crime_hist.drop(columns=overlap)

In [7]:
crime = pd.concat([crime_hist_clean, crime_recent], ignore_index=True)

In [8]:
# 只取 2021-01 到 2025-12 的月份列
target_months = [col for col in crime.columns if col.isdigit() and ("202101" <= col <= "202512")]

print("Number of target months:", len(target_months))
print(target_months[:5], "...", target_months[-5:])

Number of target months: 60
['202101', '202102', '202103', '202104', '202105'] ... ['202508', '202509', '202510', '202511', '202512']


In [9]:
print(sorted(crime["Major Category"].dropna().unique()))

['ARSON AND CRIMINAL DAMAGE', 'BURGLARY', 'DRUG OFFENCES', 'MISCELLANEOUS CRIMES AGAINST SOCIETY', 'POSSESSION OF WEAPONS', 'PUBLIC ORDER OFFENCES', 'ROBBERY', 'THEFT', 'VEHICLE OFFENCES', 'VIOLENCE AGAINST THE PERSON']


In [10]:
target_crimes = [
    "BURGLARY",
    "THEFT",
    "VIOLENCE AGAINST THE PERSON"
]

crime3 = crime[crime["Major Category"].isin(target_crimes)].copy()

print(crime3["Major Category"].value_counts())
print(crime3.shape)

Major Category
THEFT                          35714
VIOLENCE AGAINST THE PERSON    32385
BURGLARY                       31012
Name: count, dtype: int64
(99111, 197)


In [11]:
keep_cols = ["LSOA Code", "LSOA Name", "Borough", "Major Category"] + target_months
crime3 = crime3[keep_cols].copy()

In [12]:
crime3[target_months] = crime3[target_months].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

In [13]:
crime3["total_count"] = crime3[target_months].sum(axis=1)

In [16]:
crime_sum = (
    crime_sum.groupby(["LSOA Code", "Major Category"])["total_count"]
    .sum()
    .reset_index()
)

In [17]:
crime_wide = crime_sum.pivot(
    index="LSOA Code",
    columns="Major Category",
    values="total_count"
).reset_index()

crime_wide.columns.name = None
print(crime_wide.head())

   LSOA Code  BURGLARY  THEFT  VIOLENCE AGAINST THE PERSON
0  E01000006      15.0   62.0                        157.0
1  E01000007      76.0  658.0                        863.0
2  E01000008      46.0  164.0                        302.0
3  E01000009      62.0  275.0                        459.0
4  E01000011      23.0   63.0                        286.0


In [18]:
crime_wide = crime_wide.rename(columns={
    "LSOA Code": "LSOA_code",
    "BURGLARY": "burglary",
    "THEFT": "theft",
    "VIOLENCE AGAINST THE PERSON": "violence"
})

In [19]:
years_observed = len(target_months) / 12
print("Months:", len(target_months))
print("Years observed:", years_observed)

Months: 60
Years observed: 5.0


In [ ]:
####简单测试

In [20]:
import pandas as pd

pop = pd.read_csv("lsoa_pop_raw.csv")

In [21]:
print(pop.shape)
print(pop.columns)
print(pop.head())

(4659, 7)
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'LSOA_code', 'population',
       'area_km2', 'pop_density'],
      dtype='object')
    LSOA11CD                   LSOA11NM               LAD11NM  LSOA_code  \
0  E01000001        City of London 001A        City of London  E01000001   
1  E01000002        City of London 001B        City of London  E01000002   
2  E01000003        City of London 001C        City of London  E01000003   
3  E01000005        City of London 001E        City of London  E01000005   
4  E01000006  Barking and Dagenham 016A  Barking and Dagenham  E01000006   

   population  area_km2   pop_density  
0      1475.0  0.133321  11063.542564  
1      1384.0  0.226191   6118.715289  
2      1613.0  0.057303  28148.629970  
3      1100.0  0.190739   5767.050167  
4      1845.0  0.144196  12795.098065  


In [22]:
pop = pop[["LSOA_code", "population"]].copy()

In [23]:
df = pop.merge(crime_wide, on="LSOA_code", how="left")

In [24]:
for col in ["burglary", "theft", "violence"]:
    df[col] = df[col].fillna(0)

In [25]:
df["burglary_annual"] = df["burglary"] / years_observed
df["theft_annual"] = df["theft"] / years_observed
df["violence_annual"] = df["violence"] / years_observed

In [26]:
df["burglary_rate"] = df["burglary_annual"] / df["population"] * 1000
df["theft_rate"] = df["theft_annual"] / df["population"] * 1000
df["violence_rate"] = df["violence_annual"] / df["population"] * 1000

In [27]:
print(df[[
    "LSOA_code",
    "population",
    "burglary_rate",
    "theft_rate",
    "violence_rate"
]].head())

print(df[["burglary_rate", "theft_rate", "violence_rate"]].describe())

   LSOA_code  population  burglary_rate  theft_rate  violence_rate
0  E01000001      1475.0       0.000000    0.000000        0.00000
1  E01000002      1384.0       0.000000    0.000000        0.00000
2  E01000003      1613.0       0.000000    0.000000        0.00000
3  E01000005      1100.0       0.000000    0.000000        0.00000
4  E01000006      1845.0       1.626016    6.720867       17.01897
       burglary_rate   theft_rate  violence_rate
count    4659.000000  4659.000000    4659.000000
mean        5.599537    25.249950      24.926563
std         4.012864    94.092439      22.107819
min         0.000000     0.000000       0.000000
25%         3.320032     5.638542      14.218257
50%         4.705882     9.766926      21.453287
75%         6.733405    20.428556      29.219952
max        63.630988  3397.222222     675.458309


In [29]:
df.to_csv("crime_rates_lsoa_2021_2025.csv", index=False)

In [32]:
import pandas as pd

food = pd.read_csv("dataset/lsoa_food.csv")
shop = pd.read_csv("dataset/lsoa_shop.csv")
night = pd.read_csv("dataset/lsoa_night.csv")

In [33]:
print(food.columns)
print(shop.columns)
print(night.columns)

Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'food_poi_count', 'area_km2',
       'food_poi_density'],
      dtype='object')
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'shop_poi_count', 'area_km2',
       'shop_poi_density'],
      dtype='object')
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'night_poi_count', 'area_km2',
       'night_poi_density'],
      dtype='object')


In [34]:
food.head()
shop.head()
night.head()

,LSOA11CD,LSOA11NM,LAD11NM,night_poi_count,area_km2,night_poi_density
0,E01000001,City of London 001A,City of London,4.0,0.133321,30.002827
1,E01000002,City of London 001B,City of London,5.0,0.226191,22.105185
2,E01000003,City of London 001C,City of London,1.0,0.057303,17.451104
3,E01000005,City of London 001E,City of London,19.0,0.190739,99.612685
4,E01000006,Barking and Dagenham 016A,Barking and Dagenham,0.0,0.144196,0.000000


In [35]:
print("FOOD")
print(food.describe())

print("\nSHOP")
print(shop.describe())

print("\nNIGHT")
print(night.describe())

FOOD
       food_poi_count     area_km2  food_poi_density
count     4835.000000  4835.000000       4835.000000
mean         4.508790     0.325441         23.890635
std         14.397041     0.629009         55.779096
min          0.000000     0.016902          0.000000
25%          0.000000     0.133548          0.000000
50%          1.000000     0.203355          3.691477
75%          4.000000     0.319056         22.448862
max        553.000000    15.808727       1002.161471

SHOP
       shop_poi_count     area_km2  shop_poi_density
count     4835.000000  4835.000000       4835.000000
mean         8.865357     0.325441         48.106087
std         21.499340     0.629009         99.806246
min          0.000000     0.016902          0.000000
25%          0.000000     0.133548          0.000000
50%          2.000000     0.203355          9.766177
75%          9.000000     0.319056         48.444523
max        606.000000    15.808727       1460.367160

NIGHT
       night_poi_count     a

In [ ]:
###########

In [ ]:
########合成大表

In [37]:
import pandas as pd

# Y变量
crime = pd.read_csv("dataset/crime_rates_lsoa_2021_2025.csv")

# 保证字段一致
crime = crime.rename(columns={"LSOA_code": "LSOA_code"})
print(crime.shape)
crime.head()

(4659, 11)


,LSOA_code,population,burglary,theft,violence,burglary_annual,theft_annual,violence_annual,burglary_rate,theft_rate,violence_rate
0,E01000001,1475.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000
1,E01000002,1384.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000
2,E01000003,1613.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000
3,E01000005,1100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000
4,E01000006,1845.0,15.0,62.0,157.0,3.0,12.4,31.4,1.626016,6.720867,17.01897


In [39]:
# population + density
pop = pd.read_csv("dataset/lsoa_pop_raw.csv")

# workplace density
work = pd.read_csv("dataset/lsoa_workplacepopulation.csv")

# POI
food = pd.read_csv("dataset/lsoa_food.csv")
shop = pd.read_csv("dataset/lsoa_shop.csv")
night = pd.read_csv("dataset/lsoa_night.csv")

# network
bet = pd.read_csv("dataset/lsoa_betweenness.csv")
inter = pd.read_csv("dataset/lsoa_with_intersection_density.csv")

In [40]:
imd = pd.read_csv("dataset/imd_lsoa.csv")

print(imd.columns)
imd.head()

Index(['LSOA_code', 'imd_score'], dtype='object')


,LSOA_code,imd_score
0,E01000001,3645
1,E01000002,2465
2,E01000003,17929
3,E01000005,24166
4,E01000006,18358


In [41]:
imd = imd.rename(columns={
    "LSOA Code": "LSOA_code",
    "IMD Score": "IMD_score"
})

In [42]:
for df_ in [pop, work, food, shop, night, bet, inter]:
    print(df_.columns)

Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'LSOA_code', 'population',
       'area_km2', 'pop_density'],
      dtype='object')
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'LSOA_code_x', 'population',
       'area_km2', 'pop_density', 'LSOA_code_y', 'workplace_pop',
       'workplace_density'],
      dtype='object')
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'food_poi_count', 'area_km2',
       'food_poi_density'],
      dtype='object')
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'shop_poi_count', 'area_km2',
       'shop_poi_density'],
      dtype='object')
Index(['LSOA11CD', 'LSOA11NM', 'LAD11NM', 'night_poi_count', 'area_km2',
       'night_poi_density'],
      dtype='object')
Index(['LSOA11CD', 'betweenness'], dtype='object')
Index(['LSOA11CD', 'area_km2', 'intersection_count', 'intersection_density'], dtype='object')


In [45]:
pop = pop.rename(columns={"LSOA11CD": "LSOA_code"})
work = work.rename(columns={"LSOA11CD": "LSOA_code"})
food = food.rename(columns={"LSOA11CD": "LSOA_code"})
shop = shop.rename(columns={"LSOA11CD": "LSOA_code"})
night = night.rename(columns={"LSOA11CD": "LSOA_code"})
bet = bet.rename(columns={"LSOA11CD": "LSOA_code"})
inter = inter.rename(columns={"LSOA11CD": "LSOA_code"})

In [46]:
pop = pop[["LSOA_code", "population", "pop_density"]]

work = work[["LSOA_code", "workplace_density"]]

food = food[["LSOA_code", "food_poi_density"]]

shop = shop[["LSOA_code", "shop_poi_density"]]

night = night[["LSOA_code", "night_poi_density"]]

bet = bet[["LSOA_code", "betweenness"]]

inter = inter[["LSOA_code", "intersection_density"]]

In [47]:
df = pop.copy()

df = df.merge(work, on="LSOA_code", how="left")
df = df.merge(food, on="LSOA_code", how="left")
df = df.merge(shop, on="LSOA_code", how="left")
df = df.merge(night, on="LSOA_code", how="left")
df = df.merge(bet, on="LSOA_code", how="left")
df = df.merge(inter, on="LSOA_code", how="left")

ValueError: The column label 'LSOA_code' is not unique.

In [48]:
for name, d in {
    "pop": pop,
    "work": work,
    "food": food,
    "shop": shop,
    "night": night,
    "bet": bet,
    "inter": inter
}.items():
    dup_cols = d.columns[d.columns.duplicated()].tolist()
    print(name, dup_cols)
    print(d.columns.tolist())
    print("------")

pop ['LSOA_code']
['LSOA_code', 'LSOA_code', 'population', 'pop_density']
------
work []
['LSOA_code', 'workplace_density']
------
food []
['LSOA_code', 'food_poi_density']
------
shop []
['LSOA_code', 'shop_poi_density']
------
night []
['LSOA_code', 'night_poi_density']
------
bet []
['LSOA_code', 'betweenness']
------
inter []
['LSOA_code', 'intersection_density']
------


In [49]:
pop = pop.loc[:, ~pop.columns.duplicated()].copy()
print(pop.columns.tolist())

['LSOA_code', 'population', 'pop_density']


In [50]:
print(pop.columns.tolist())
print(pop.columns.duplicated().sum())

['LSOA_code', 'population', 'pop_density']
0


In [51]:
final_df = df.copy()

final_df = final_df.merge(pop, on="LSOA_code", how="left")
final_df = final_df.merge(work, on="LSOA_code", how="left")
final_df = final_df.merge(food, on="LSOA_code", how="left")
final_df = final_df.merge(shop, on="LSOA_code", how="left")
final_df = final_df.merge(night, on="LSOA_code", how="left")
final_df = final_df.merge(bet, on="LSOA_code", how="left")
final_df = final_df.merge(inter, on="LSOA_code", how="left")

ValueError: The column label 'LSOA_code' is not unique.

In [53]:
print("df duplicated cols:", df.columns[df.columns.duplicated()].tolist())
print("pop duplicated cols:", pop.columns[pop.columns.duplicated()].tolist())
print("work duplicated cols:", work.columns[work.columns.duplicated()].tolist())
print("food duplicated cols:", food.columns[food.columns.duplicated()].tolist())
print("shop duplicated cols:", shop.columns[shop.columns.duplicated()].tolist())
print("night duplicated cols:", night.columns[night.columns.duplicated()].tolist())
print("bet duplicated cols:", bet.columns[bet.columns.duplicated()].tolist())
print("inter duplicated cols:", inter.columns[inter.columns.duplicated()].tolist())

df duplicated cols: ['LSOA_code']
pop duplicated cols: []
work duplicated cols: []
food duplicated cols: []
shop duplicated cols: []
night duplicated cols: []
bet duplicated cols: []
inter duplicated cols: []


In [54]:
df = df.loc[:, ~df.columns.duplicated()].copy()
pop = pop.loc[:, ~pop.columns.duplicated()].copy()
work = work.loc[:, ~work.columns.duplicated()].copy()
food = food.loc[:, ~food.columns.duplicated()].copy()
shop = shop.loc[:, ~shop.columns.duplicated()].copy()
night = night.loc[:, ~night.columns.duplicated()].copy()
bet = bet.loc[:, ~bet.columns.duplicated()].copy()
inter = inter.loc[:, ~inter.columns.duplicated()].copy()

In [55]:
print("df duplicated cols:", df.columns[df.columns.duplicated()].tolist())
print("pop duplicated cols:", pop.columns[pop.columns.duplicated()].tolist())
print("work duplicated cols:", work.columns[work.columns.duplicated()].tolist())
print("food duplicated cols:", food.columns[food.columns.duplicated()].tolist())
print("shop duplicated cols:", shop.columns[shop.columns.duplicated()].tolist())
print("night duplicated cols:", night.columns[night.columns.duplicated()].tolist())
print("bet duplicated cols:", bet.columns[bet.columns.duplicated()].tolist())
print("inter duplicated cols:", inter.columns[inter.columns.duplicated()].tolist())

df duplicated cols: []
pop duplicated cols: []
work duplicated cols: []
food duplicated cols: []
shop duplicated cols: []
night duplicated cols: []
bet duplicated cols: []
inter duplicated cols: []


In [56]:
final_df = df.copy()

final_df = final_df.merge(pop, on="LSOA_code", how="left")
final_df = final_df.merge(work, on="LSOA_code", how="left")
final_df = final_df.merge(food, on="LSOA_code", how="left")
final_df = final_df.merge(shop, on="LSOA_code", how="left")
final_df = final_df.merge(night, on="LSOA_code", how="left")
final_df = final_df.merge(bet, on="LSOA_code", how="left")
final_df = final_df.merge(inter, on="LSOA_code", how="left")

In [57]:
print(final_df.shape)
print(final_df.isna().sum().sort_values(ascending=False))
print(final_df.head())

(4659, 11)
LSOA_code               0
population_x            0
pop_density_x           0
population_y            0
pop_density_y           0
workplace_density       0
food_poi_density        0
shop_poi_density        0
night_poi_density       0
betweenness             0
intersection_density    0
dtype: int64
   LSOA_code  population_x  pop_density_x  population_y  pop_density_y  \
0  E01000001        1475.0   11063.542564        1475.0   11063.542564   
1  E01000002        1384.0    6118.715289        1384.0    6118.715289   
2  E01000003        1613.0   28148.629970        1613.0   28148.629970   
3  E01000005        1100.0    5767.050167        1100.0    5767.050167   
4  E01000006        1845.0   12795.098065        1845.0   12795.098065   

   workplace_density  food_poi_density  shop_poi_density  night_poi_density  \
0       24422.301415        105.009896         67.506361          30.002827   
1       31964.097935         88.420741         22.105185          22.105185   
2       

In [58]:
final_df = final_df.drop(columns=["population_y", "pop_density_y"])
final_df = final_df.rename(columns={
    "population_x": "population",
    "pop_density_x": "pop_density"
})

In [59]:
print(final_df.columns.tolist())
print(final_df.shape)
print(final_df.isna().sum().sum())
final_df.head()

['LSOA_code', 'population', 'pop_density', 'workplace_density', 'food_poi_density', 'shop_poi_density', 'night_poi_density', 'betweenness', 'intersection_density']
(4659, 9)
0


,LSOA_code,population,pop_density,workplace_density,food_poi_density,shop_poi_density,night_poi_density,betweenness,intersection_density
0,E01000001,1475.0,11063.542564,24422.301415,105.009896,67.506361,30.002827,0.005040,172.516257
1,E01000002,1384.0,6118.715289,31964.097935,88.420741,22.105185,22.105185,0.001856,114.946963
2,E01000003,1613.0,28148.629970,15077.753437,34.902207,122.157725,17.451104,0.000692,87.255518
3,E01000005,1100.0,5767.050167,18040.381477,319.809146,214.953688,99.612685,0.006598,204.468142
4,E01000006,1845.0,12795.098065,2926.575276,0.000000,0.000000,0.000000,0.008735,69.350125


In [60]:
print((final_df["food_poi_density"] == 0).mean())
print((final_df["shop_poi_density"] == 0).mean())
print((final_df["night_poi_density"] == 0).mean())

0.4455891822279459
0.3326894183301138
0.6512127065893969


In [61]:
print(final_df.columns.tolist())
print(final_df.shape)
print(final_df.isna().sum().sum())
final_df.head()

['LSOA_code', 'population', 'pop_density', 'workplace_density', 'food_poi_density', 'shop_poi_density', 'night_poi_density', 'betweenness', 'intersection_density']
(4659, 9)
0


,LSOA_code,population,pop_density,workplace_density,food_poi_density,shop_poi_density,night_poi_density,betweenness,intersection_density
0,E01000001,1475.0,11063.542564,24422.301415,105.009896,67.506361,30.002827,0.005040,172.516257
1,E01000002,1384.0,6118.715289,31964.097935,88.420741,22.105185,22.105185,0.001856,114.946963
2,E01000003,1613.0,28148.629970,15077.753437,34.902207,122.157725,17.451104,0.000692,87.255518
3,E01000005,1100.0,5767.050167,18040.381477,319.809146,214.953688,99.612685,0.006598,204.468142
4,E01000006,1845.0,12795.098065,2926.575276,0.000000,0.000000,0.000000,0.008735,69.350125


In [62]:
final_df.to_csv("final_dataset.csv", index=False)